# Solutions — Intro to PySpark (optional — needs pyspark + Java)

Exercise notebook: `12_pyspark_preparation/04_intro_pyspark.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
# Optional section: needs `pip install pyspark` and Java 17+ (see SETUP.md). Everything else in the lab works without it.
import os
os.environ.setdefault("PYSPARK_PYTHON", "python")
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (SparkSession.builder.master("local[2]").appName("python-hands-on-lab")
         .config("spark.sql.shuffle.partitions", "4").config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(spark.version)

## Exercise 1 — 🟢 Easy · Read CSV with an explicit schema

**Working solution**

In [ ]:
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

ORDER_SCHEMA = StructType([
    StructField("order_id", StringType()), StructField("customer_id", StringType()), StructField("product_id", StringType()),
    StructField("quantity", IntegerType()), StructField("unit_price", DoubleType()), StructField("order_ts", StringType()),
    StructField("status", StringType()), StructField("updated_at", StringType()), StructField("channel", StringType()),
    StructField("country", StringType()),
])

def read_orders(spark, path):
    return spark.read.csv(str(path), header=True, schema=ORDER_SCHEMA, mode="PERMISSIVE")

**Explanation**

An explicit schema skips the inference pass over the data, pins types (no surprise strings), and documents the contract. `PERMISSIVE` + nulls lets you count bad values instead of crashing.

**Complexity:** Lazy; the scan happens on the first action.

**Common mistakes**

- `inferSchema=True` on production data — slow and unstable.

## Exercise 2 — 🟡 Medium · Clean and keep the latest version with a window

**Working solution**

In [ ]:
from pyspark.sql import Window, functions as F

def clean_orders(df):
    ts_fmt = "yyyy-MM-dd'T'HH:mm:ss'Z'"
    cleaned = (df.withColumn("status", F.upper(F.trim("status")))
                 .withColumn("channel", F.lower("channel"))
                 .withColumn("order_ts", F.to_timestamp("order_ts", ts_fmt))
                 .withColumn("updated_at", F.to_timestamp("updated_at", ts_fmt))
                 .filter((F.col("quantity") >= 1) & F.col("unit_price").isNotNull() & (F.col("unit_price") >= 0)
                         & F.col("customer_id").isNotNull() & F.col("order_ts").isNotNull())
                 .withColumn("amount", F.round(F.col("quantity") * F.col("unit_price"), 2)))
    w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
    return cleaned.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

**Explanation**

The same 'latest version wins' you wrote with dicts (Level 8) and `drop_duplicates` (Level 9), now as a window function — Spark shuffles by `order_id` and sorts within each partition.

**Complexity:** One shuffle for the window.

**Common mistakes**

- `dropDuplicates(['order_id'])` — keeps an *arbitrary* row, not the latest.

## Exercise 3 — 🟠 Hard · Aggregate daily revenue

**Working solution**

In [ ]:
from pyspark.sql import functions as F

def daily_revenue(df):
    return (df.filter(F.col("status") != "CANCELLED")
              .groupBy(F.to_date("order_ts").alias("order_date"), "country")
              .agg(F.countDistinct("order_id").alias("orders"), F.sum("quantity").alias("units"), F.round(F.sum("amount"), 2).alias("revenue"))
              .orderBy("order_date", "country"))

**Explanation**

`groupBy().agg()` with named aliases is the Spark twin of pandas named aggregation. `countDistinct` is expensive (it can't be partially combined as cheaply) — use `approx_count_distinct` on huge data.

**Complexity:** One shuffle (+ one for orderBy).

**Common mistakes**

- Collecting to the driver to aggregate in Python.

## Exercise 4 — 🔴 Very Hard · Partitioned, idempotent writes

**Working solution**

In [ ]:
from pyspark.sql import functions as F

def write_daily(df, path):
    spark = df.sparkSession
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    (df.withColumn("order_date", F.to_date("order_ts"))
       .write.mode("overwrite")
       .partitionBy("order_date")
       .parquet(str(path)))

**Explanation**

With the default *static* mode, `overwrite` deletes the **whole table directory** first — rerunning one day wipes the other days. Dynamic mode replaces only partitions present in the
DataFrame being written, which makes per-day reruns idempotent and safe. (Delta/Iceberg offer `replaceWhere`/`MERGE` for the same goal with ACID guarantees.)

**Complexity:** One write job; files per partition ≈ number of tasks holding that partition's rows.

**Common mistakes**

- Static overwrite on a partitioned table → data loss on reruns.
- Thousands of tiny files: `repartition('order_date')` before writing.